<a href="https://colab.research.google.com/github/arcctg/kpi-ml-lab6/blob/main/01_applicant_admission_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applicant Admission Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Data Generation

In [ ]:
np.random.seed(42)

N = 1500
PRIVILEGED_RATIO = 0.13
N_ANOMALIES = 50

math_scores = np.clip(np.random.normal(155, 20, N), 100, 200).astype(int)
english_scores = np.clip(np.random.normal(155, 20, N), 100, 200).astype(int)
ukrainian_scores = np.clip(np.random.normal(155, 20, N), 100, 200).astype(int)

privileged = np.zeros(N, dtype=int)
privileged_indices = np.random.choice(N, size=int(N * PRIVILEGED_RATIO), replace=False)
privileged[privileged_indices] = 1

anomaly_indices = np.random.choice(N, size=N_ANOMALIES, replace=False)
for i, idx in enumerate(anomaly_indices):
    anomaly_type = i % 5
    if anomaly_type == 0:
        math_scores[idx] = np.random.randint(190, 201)
        english_scores[idx] = np.random.randint(180, 201)
        ukrainian_scores[idx] = np.random.randint(100, 115)
    elif anomaly_type == 1:
        math_scores[idx] = np.random.randint(100, 115)
        english_scores[idx] = np.random.randint(100, 115)
        ukrainian_scores[idx] = np.random.randint(100, 115)
    elif anomaly_type == 2:
        math_scores[idx] = np.random.randint(100, 130)
        english_scores[idx] = np.random.randint(190, 201)
        ukrainian_scores[idx] = np.random.randint(170, 201)
    elif anomaly_type == 3:
        math_scores[idx] = 200
        english_scores[idx] = 200
        ukrainian_scores[idx] = 200
    elif anomaly_type == 4:
        math_scores[idx] = np.random.randint(180, 201)
        english_scores[idx] = np.random.randint(100, 115)
        ukrainian_scores[idx] = np.random.randint(180, 201)

rating = 0.4 * math_scores + 0.3 * english_scores + 0.3 * ukrainian_scores

df = pd.DataFrame({
    'id': range(1, N + 1),
    'math': math_scores,
    'english': english_scores,
    'ukrainian': ukrainian_scores,
    'privileged': privileged,
    'rating': rating
})

print(f"Generated {N} applicants")
print(f"Privileged: {privileged.sum()} ({privileged.sum()/N*100:.1f}%)")
print(f"Anomalous cases injected: {N_ANOMALIES}")

In [ ]:
def determine_admission(df, total_seats=350, privileged_quota=35):
    admitted = np.zeros(len(df), dtype=int)

    priv_mask = df['privileged'] == 1
    non_priv_mask = df['privileged'] == 0

    priv_eligible = df[priv_mask &
                       (df['math'] >= 120) &
                       (df['english'] >= 120) &
                       (df['ukrainian'] >= 120) &
                       (df['rating'] >= 144)].copy()

    non_priv_eligible = df[non_priv_mask &
                           (df['math'] >= 140) &
                           (df['english'] >= 120) &
                           (df['ukrainian'] >= 120) &
                           (df['rating'] >= 160)].copy()

    priv_eligible = priv_eligible.sort_values('rating', ascending=False)
    non_priv_eligible = non_priv_eligible.sort_values('rating', ascending=False)

    priv_admitted = priv_eligible.head(privileged_quota)
    n_priv_admitted = len(priv_admitted)

    remaining_seats = total_seats - n_priv_admitted
    non_priv_admitted = non_priv_eligible.head(remaining_seats)

    admitted_indices = list(priv_admitted.index) + list(non_priv_admitted.index)
    admitted[admitted_indices] = 1

    return admitted

df['admitted'] = determine_admission(df)

os.makedirs('data', exist_ok=True)
df.to_csv('data/applicants.csv', index=False)

print(f"Dataset shape: {df.shape}")
print(f"Admitted: {df['admitted'].sum()}")
print(f"Rejected: {(df['admitted'] == 0).sum()}")
print(f"Privileged total: {df['privileged'].sum()}")
print(f"Privileged admitted: {df[(df['privileged'] == 1) & (df['admitted'] == 1)].shape[0]}")
print(f"Non-privileged admitted: {df[(df['privileged'] == 0) & (df['admitted'] == 1)].shape[0]}")

## 2. Exploratory Data Analysis

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, color in zip(axes, ['math', 'english', 'ukrainian'],
                           ['#2196F3', '#4CAF50', '#FF9800']):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{col.capitalize()} Score Distribution')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.axvline(x=df[col].mean(), color='red', linestyle='--',
               label=f'Mean: {df[col].mean():.1f}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df[df['admitted'] == 1]['rating'], bins=30, alpha=0.7,
        color='#4CAF50', label='Admitted', edgecolor='white')
ax.hist(df[df['admitted'] == 0]['rating'], bins=30, alpha=0.7,
        color='#F44336', label='Rejected', edgecolor='white')
ax.set_title('Rating Distribution: Admitted vs Rejected')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
ax.axvline(x=160, color='orange', linestyle='--', linewidth=2,
           label='Min rating (non-privileged): 160')
ax.axvline(x=144, color='purple', linestyle='--', linewidth=2,
           label='Min rating (privileged): 144')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

priv_counts = df['privileged'].value_counts()
axes[0].pie(priv_counts, labels=['Non-privileged', 'Privileged'],
            autopct='%1.1f%%', colors=['#2196F3', '#FF9800'], startangle=90)
axes[0].set_title('Overall: Privileged vs Non-privileged')

admitted_priv = df[df['admitted'] == 1]['privileged'].value_counts()
axes[1].pie(admitted_priv, labels=['Non-privileged', 'Privileged'],
            autopct='%1.1f%%', colors=['#2196F3', '#FF9800'], startangle=90)
axes[1].set_title('Admitted: Privileged vs Non-privileged')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

correlation = df[['math', 'english', 'ukrainian', 'privileged',
                   'rating', 'admitted']].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', ax=ax)
ax.set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(df['rating'], df['math'], c=df['admitted'],
                     cmap='RdYlGn', alpha=0.5, edgecolors='none')
ax.set_xlabel('Rating')
ax.set_ylabel('Math Score')
ax.set_title('Rating vs Math Score (colored by admission)')
ax.axvline(x=160, color='orange', linestyle='--', alpha=0.7,
           label='Min rating: 160')
ax.axhline(y=140, color='red', linestyle='--', alpha=0.7,
           label='Min math: 140')
plt.colorbar(scatter, label='Admitted')
ax.legend()

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
features = ['math', 'english', 'ukrainian', 'privileged']
target = 'admitted'

X = df[features].values
y = df[target].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {features}")

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("Before scaling:")
print(f"  Min: {X.min(axis=0)}")
print(f"  Max: {X.max(axis=0)}")
print(f"\nAfter scaling:")
print(f"  Min: {X_scaled.min(axis=0)}")
print(f"  Max: {X_scaled.max(axis=0)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(f"  Admitted: {y_train.sum()} ({y_train.sum()/len(y_train)*100:.1f}%)")
print(f"  Rejected: {(y_train == 0).sum()} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")
print(f"\nTest class distribution:")
print(f"  Admitted: {y_test.sum()} ({y_test.sum()/len(y_test)*100:.1f}%)")
print(f"  Rejected: {(y_test == 0).sum()} ({(y_test == 0).sum()/len(y_test)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, data, title in zip(axes, [y_train, y_test], ['Training Set', 'Test Set']):
    counts = [np.sum(data == 0), np.sum(data == 1)]
    bars = ax.bar(['Rejected', 'Admitted'], counts,
                  color=['#F44336', '#4CAF50'], edgecolor='white')
    ax.set_title(f'{title} Class Distribution')
    ax.set_ylabel('Count')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                f'{count}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Neural Network Models

In [ ]:
def build_model(architecture, optimizer='adam', input_dim=4):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for units, activation in architecture:
        model.add(layers.Dense(units, activation=activation))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_model(model, X_train, y_train, X_test, y_test,
                epochs=100, batch_size=32, verbose=0):
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test),
        verbose=verbose
    )
    training_time = time.time() - start_time
    return history, training_time

def plot_training_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['accuracy'], label='Train Accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

model_results = {}

### Model 1: Minimal Network (1 hidden layer, 8 neurons)

In [ ]:
tf.random.set_seed(42)

model1_arch = [(8, 'sigmoid')]
model1 = build_model(model1_arch, optimizer='adam')
model1.summary()

In [ ]:
history1, time1 = train_model(model1, X_train, y_train, X_test, y_test,
                               epochs=100, batch_size=32)

loss1, acc1 = model1.evaluate(X_test, y_test, verbose=0)
print(f"Model 1 - Test Loss: {loss1:.4f}, Test Accuracy: {acc1:.4f}")
print(f"Training time: {time1:.2f}s")

model_results['Model 1 (8-sigmoid)'] = {
    'accuracy': acc1, 'loss': loss1, 'time': time1,
    'params': model1.count_params()
}

In [ ]:
plot_training_history(history1, 'Model 1: Minimal (8-sigmoid)')

### Model 2: Medium Network (2 hidden layers, 16-8 neurons)

In [ ]:
tf.random.set_seed(42)

model2_arch = [(16, 'relu'), (8, 'relu')]
model2 = build_model(model2_arch, optimizer='adam')
model2.summary()

In [ ]:
history2, time2 = train_model(model2, X_train, y_train, X_test, y_test,
                               epochs=100, batch_size=32)

loss2, acc2 = model2.evaluate(X_test, y_test, verbose=0)
print(f"Model 2 - Test Loss: {loss2:.4f}, Test Accuracy: {acc2:.4f}")
print(f"Training time: {time2:.2f}s")

model_results['Model 2 (16-8-relu)'] = {
    'accuracy': acc2, 'loss': loss2, 'time': time2,
    'params': model2.count_params()
}

In [ ]:
plot_training_history(history2, 'Model 2: Medium (16-8-relu)')

### Model 3: Extended Network (3 hidden layers, 32-16-8 neurons + Dropout)

In [ ]:
tf.random.set_seed(42)

model3 = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model3.summary()

In [ ]:
history3, time3 = train_model(model3, X_train, y_train, X_test, y_test,
                               epochs=100, batch_size=32)

loss3, acc3 = model3.evaluate(X_test, y_test, verbose=0)
print(f"Model 3 - Test Loss: {loss3:.4f}, Test Accuracy: {acc3:.4f}")
print(f"Training time: {time3:.2f}s")

model_results['Model 3 (32-16-8-relu-dropout)'] = {
    'accuracy': acc3, 'loss': loss3, 'time': time3,
    'params': model3.count_params()
}

In [ ]:
plot_training_history(history3, 'Model 3: Extended (32-16-8-relu-dropout)')

### Architecture Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for h, name in [(history1, 'Model 1'), (history2, 'Model 2'), (history3, 'Model 3')]:
    axes[0].plot(h.history['val_loss'], label=name)
    axes[1].plot(h.history['val_accuracy'], label=name)

axes[0].set_title('Validation Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
arch_comparison = pd.DataFrame(model_results).T
arch_comparison = arch_comparison.round(4)
print("Architecture Comparison:")
arch_comparison

## 5. Optimizer Comparison

In [ ]:
optimizers = {
    'SGD': 'sgd',
    'Adagrad': 'adagrad',
    'Adadelta': 'adadelta',
    'RMSProp': 'rmsprop',
    'Adam': 'adam'
}

best_architecture = [(16, 'relu'), (8, 'relu')]

optimizer_histories = {}
optimizer_results = {}

for name, opt in optimizers.items():
    print(f"\nTraining with {name}...")
    tf.random.set_seed(42)

    model = build_model(best_architecture, optimizer=opt)
    history, train_time = train_model(model, X_train, y_train, X_test, y_test,
                                      epochs=100, batch_size=32)

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"  {name} - Accuracy: {acc:.4f}, Loss: {loss:.4f}, Time: {train_time:.2f}s")

    optimizer_histories[name] = history
    optimizer_results[name] = {
        'accuracy': acc,
        'loss': loss,
        'time': train_time,
        'final_train_acc': history.history['accuracy'][-1],
        'final_val_acc': history.history['val_accuracy'][-1]
    }

    if name == 'Adam':
        best_model = model

print("\nAll optimizers trained!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#F44336', '#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for (name, h), color in zip(optimizer_histories.items(), colors):
    axes[0].plot(h.history['val_loss'], label=name, color=color)
    axes[1].plot(h.history['val_accuracy'], label=name, color=color)

axes[0].set_title('Validation Loss by Optimizer')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy by Optimizer')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
opt_df = pd.DataFrame(optimizer_results).T
opt_df = opt_df.round(4)
opt_df.index.name = 'Optimizer'
print("Optimizer Comparison:")
opt_df

## 6. Model Evaluation

In [ ]:
y_pred_prob = best_model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Admitted']))

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Rejected', 'Admitted'],
            yticklabels=['Rejected', 'Admitted'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (correctly rejected): {tn}")
print(f"False Positives (wrongly admitted): {fp}")
print(f"False Negatives (wrongly rejected): {fn}")
print(f"True Positives (correctly admitted): {tp}")

### Error Analysis

In [ ]:
test_df = df.iloc[y_test.astype(bool).tolist() + (y_test == 0).tolist()]

X_test_original = scaler.inverse_transform(X_test)
test_analysis = pd.DataFrame(X_test_original, columns=features)
test_analysis['actual'] = y_test
test_analysis['predicted'] = y_pred
test_analysis['rating'] = 0.4 * test_analysis['math'] + 0.3 * test_analysis['english'] + 0.3 * test_analysis['ukrainian']

false_positives = test_analysis[(test_analysis['actual'] == 0) & (test_analysis['predicted'] == 1)]
false_negatives = test_analysis[(test_analysis['actual'] == 1) & (test_analysis['predicted'] == 0)]

print(f"False Positives (wrongly admitted): {len(false_positives)}")
if len(false_positives) > 0:
    print(false_positives[['math', 'english', 'ukrainian', 'privileged', 'rating']].to_string())

print(f"\nFalse Negatives (wrongly rejected): {len(false_negatives)}")
if len(false_negatives) > 0:
    print(false_negatives[['math', 'english', 'ukrainian', 'privileged', 'rating']].to_string())

### Admission Count Deviation

In [ ]:
y_pred_full_prob = best_model.predict(X_scaled)
y_pred_full = (y_pred_full_prob > 0.5).astype(int).flatten()

predicted_admitted = y_pred_full.sum()
actual_admitted = y.sum()
deviation = predicted_admitted - actual_admitted

print(f"Expected admitted: {actual_admitted}")
print(f"Model predicted admitted: {predicted_admitted}")
print(f"Deviation: {deviation} ({deviation/actual_admitted*100:+.1f}%)")
print(f"\nDeviation is {'critical' if abs(deviation) > 35 else 'acceptable'}")

## 7. Anomalous Data Testing

In [ ]:
anomalous_cases = pd.DataFrame([
    {'case': 'All max scores', 'math': 200, 'english': 200, 'ukrainian': 200, 'privileged': 0,
     'expected': 'admitted'},
    {'case': 'All min scores', 'math': 100, 'english': 100, 'ukrainian': 100, 'privileged': 0,
     'expected': 'rejected'},
    {'case': 'High math+eng, low ukr', 'math': 200, 'english': 200, 'ukrainian': 100, 'privileged': 0,
     'expected': 'rejected (ukr<120)'},
    {'case': 'Privileged at boundary', 'math': 120, 'english': 120, 'ukrainian': 120, 'privileged': 1,
     'expected': 'rejected (rating=120<144)'},
    {'case': 'Privileged good scores', 'math': 160, 'english': 160, 'ukrainian': 160, 'privileged': 1,
     'expected': 'admitted'},
    {'case': 'No priv, math<140', 'math': 130, 'english': 200, 'ukrainian': 200, 'privileged': 0,
     'expected': 'rejected (math<140)'},
    {'case': 'Rating boundary 160', 'math': 170, 'english': 155, 'ukrainian': 155, 'privileged': 0,
     'expected': 'admitted (rating=161)'},
    {'case': 'Just below 160 rating', 'math': 165, 'english': 153, 'ukrainian': 153, 'privileged': 0,
     'expected': 'rejected (rating=157.8)'},
    {'case': 'High scores, low eng', 'math': 195, 'english': 105, 'ukrainian': 190, 'privileged': 0,
     'expected': 'rejected (eng<120)'},
    {'case': 'All 150 no priv', 'math': 150, 'english': 150, 'ukrainian': 150, 'privileged': 0,
     'expected': 'rejected (math<140 and rating=150<160)'},
])

anomalous_cases['rating'] = (0.4 * anomalous_cases['math'] +
                              0.3 * anomalous_cases['english'] +
                              0.3 * anomalous_cases['ukrainian'])
anomalous_cases

In [ ]:
X_anomalous = anomalous_cases[features].values
X_anomalous_scaled = scaler.transform(X_anomalous)

predictions = best_model.predict(X_anomalous_scaled)
pred_labels = (predictions > 0.5).astype(int).flatten()

anomalous_cases['prediction_prob'] = predictions.flatten().round(4)
anomalous_cases['predicted'] = ['admitted' if p == 1 else 'rejected' for p in pred_labels]

result_cols = ['case', 'math', 'english', 'ukrainian', 'privileged',
               'rating', 'expected', 'predicted', 'prediction_prob']
anomalous_cases[result_cols]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

cases = anomalous_cases['case']
probs = anomalous_cases['prediction_prob']
colors = ['#4CAF50' if p > 0.5 else '#F44336' for p in probs]

bars = ax.barh(cases, probs, color=colors, edgecolor='white')
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.set_xlabel('Prediction Probability')
ax.set_title('Neural Network Predictions on Anomalous Cases')
ax.legend()

for bar, prob in zip(bars, probs):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{prob:.3f}', va='center')

plt.tight_layout()
plt.show()

## 8. Final Results and Excel Export

In [ ]:
df_full_pred = df.copy()
X_full_scaled = scaler.transform(df_full_pred[features].values)
df_full_pred['nn_prediction'] = (best_model.predict(X_full_scaled) > 0.5).astype(int).flatten()

admitted_list = df_full_pred[df_full_pred['nn_prediction'] == 1].sort_values('rating', ascending=False)
admitted_list = admitted_list[['id', 'math', 'english', 'ukrainian', 'privileged', 'rating']]

print(f"Total admitted by neural network: {len(admitted_list)}")
print(f"Privileged admitted: {admitted_list[admitted_list['privileged'] == 1].shape[0]}")
print(f"Non-privileged admitted: {admitted_list[admitted_list['privileged'] == 0].shape[0]}")

In [ ]:
admitted_list.to_excel('data/admitted_students.xlsx', index=False)
print("Admitted students list saved to data/admitted_students.xlsx")

In [ ]:
print("Top-10 admitted students:")
admitted_list.head(10)

In [ ]:
print("Bottom-10 admitted students (closest to cutoff):")
admitted_list.tail(10)

### Final Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, color_a, color_r in zip(
    axes, ['math', 'english', 'ukrainian'],
    ['#4CAF50', '#2196F3', '#FF9800'],
    ['#F44336', '#E91E63', '#FF5722']):
    parts = ax.violinplot(
        [df[df['admitted'] == 1][col], df[df['admitted'] == 0][col]],
        positions=[1, 2], showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(color_a if i == 0 else color_r)
        pc.set_alpha(0.7)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Admitted', 'Rejected'])
    ax.set_title(f'{col.capitalize()} Score Distribution')
    ax.set_ylabel('Score')

plt.tight_layout()
plt.show()

In [ ]:
print("Best model architecture:")
best_model.summary()

In [ ]:
print("="*60)
print("FINAL SUMMARY")
print("="*60)

print("\n--- Architecture Comparison ---")
arch_df = pd.DataFrame(model_results).T
print(arch_df.round(4).to_string())

print("\n--- Optimizer Comparison ---")
opt_df_final = pd.DataFrame(optimizer_results).T
print(opt_df_final.round(4).to_string())

print(f"\n--- Final Test Metrics ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

print(f"\n--- Admission Results ---")
print(f"Expected: {actual_admitted}, Predicted: {predicted_admitted}, Deviation: {deviation}")
print(f"False Positives: {fp}, False Negatives: {fn}")